In [ ]:
import pandas as pd
from PIL import Image
import torchvision
from torchvision.models import EfficientNet_B0_Weights
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm

import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import f1_score, confusion_matrix, precision_recall_curve, auc
from kornia.losses import FocalLoss


# Read the file and load into a DataFrame
ground_truth_df = pd.read_csv('/kaggle/input/ml-exercise-therapanacea/ml_exercise_therapanacea/label_train.txt', header=None, names=['label'])
ground_truth_df.index = ground_truth_df.index + 1
df_hard = pd.read_csv('/kaggle/input/ml-exercise-therapanacea/hard_to_classify_samples.csv', index_col=0)
ground_truth_df = ground_truth_df.join(df_hard, how='outer')

# Define the model, load weights, and modify the classifier
m = torchvision.models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
m.classifier = torch.nn.Linear(in_features=1280, out_features=2, bias=True)
m.to('cuda')
# Define the required transforms for these weights 
proper_transforms = EfficientNet_B0_Weights.IMAGENET1K_V1.transforms()

# Custom dataset class for our CelebA subset
class CelebASubsetDataset(Dataset):
    def __init__(self, img_folder, ground_truth_df, indices=None, transform=None, load_weights=True):
        self.img_folder = img_folder
        self.ground_truth_df = ground_truth_df
        self.transform = transform
        if indices is None:
            self.indices = self.ground_truth_df.index.tolist()
        else:
            self.indices = indices
        self.load_weights = load_weights

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img_idx = self.indices[idx]
        img_path = f"{self.img_folder}/{img_idx:06d}.jpg"
        image = Image.open(img_path).convert("RGB")
        label = self.ground_truth_df.loc[img_idx, 'label']
        if self.load_weights:  # samples that are hard to classify will be given a higher weight
            weight = 2.0 if self.ground_truth_df.loc[img_idx, 'hard_to_classify'] else 1.0
            if self.transform:
                transf_image = self.transform(image)
                return transf_image, label, weight
            return image, label, weight
        else:
            if self.transform:
                transf_image = self.transform(image)
                return transf_image, label
            return image, label


# Metric to minimize in this exercise
def compute_hter(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # avoid division by zero
    far = fp / (fp + tn + 1e-8)  # False Acceptance Rate
    frr = fn / (fn + tp + 1e-8)  # False Rejection Rate
    hter = 0.5 * (far + frr)
    return hter

In [ ]:
# For this last training run, we will use all the available images, no validation
training_ds = CelebASubsetDataset(
img_folder='/kaggle/input/ml-exercise-therapanacea/ml_exercise_therapanacea/train_img',
ground_truth_df=ground_truth_df,
indices=list(range(1, 100000)), 
transform=proper_transforms
)

# best Hyperparameters found during the previous runs
num_epochs = 4
focal_gamma = 6
focal_alpha = 0.1
batch_size = 64
learning_rate = 1e-4

# DataLoader
train_loader = DataLoader(training_ds, batch_size=batch_size, shuffle=True)

# Loss and optimizer
criterion = FocalLoss(gamma=focal_gamma, alpha=focal_alpha, reduction='none')  # reduction none because we will apply weights to our loss
optimizer = optim.Adam(m.parameters(), lr=learning_rate)

In [8]:
# Training loop
for epoch in range(num_epochs):
    m.train()
    for i, (transf_images, labels, weights) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        transf_images, labels, weights = transf_images.to('cuda'), labels.to('cuda'), weights.to('cuda')
        optimizer.zero_grad()
        outputs = m(transf_images)
        
        loss = criterion(outputs, labels)
        weighted_loss = (loss * weights.unsqueeze(1)).mean()
        weighted_loss.backward()
        optimizer.step()


Epoch 4/4: 100%|██████████| 1563/1563 [07:44<00:00,  3.36it/s]


In [ ]:
# Prepare dataset and dataloader for the Test images.
# (NB: ground_truth_df is not needed here but for the sake of running this exercise once, I didn't want to modify the dataset class)
test = CelebASubsetDataset(
    img_folder='/kaggle/input/ml-exercise-therapanacea/ml_exercise_therapanacea/val_img',
    ground_truth_df=ground_truth_df,
    indices=list(range(1,20001)),
    transform=proper_transforms,
    load_weights=False
)
val_loader = DataLoader(test, batch_size=64, shuffle=False)

# Run predictions
m.eval()
all_preds = []
with torch.no_grad():
    for images, _ in tqdm(val_loader, desc="Predicting on validation set"):  # no labels of course
        images = images.to('cuda')
        outputs = m(images)
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()  # Probability for class 1
        all_preds.extend(probs)


Predicting on validation set: 100%|██████████| 313/313 [01:17<00:00,  4.06it/s]


In [12]:
len(all_preds)

20000

In [ ]:
# export predictions to a text file as required
with open('/kaggle/working/label_val.txt', 'w') as f:
    for pred in all_preds:
        f.write(f"{pred}\n")